### Challenge 1:
Tasks:

- *Fill missing salary with the average salary per department.*
- *Convert hire_date to date type.*
- *Add a column years_with_company based on today's date.*
- *Filter only employees who have been with the company for 3+ years.*
- *Show average salary by department.*

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import Window
from pyspark.sql.types import *

In [0]:
employee_df = spark.createDataFrame([
    (101, "Alice", "HR", 58000, "2019-04-01"),
    (102, "Bob", "Engineering", 92000, "2020-07-15"),
    (103, "Charlie", "Sales", None, "2018-02-20"),
    (104, "Diana", "Engineering", 99000, None),
    (105, "Evan", "HR", 61000, "2021-09-12")
], ["emp_id", "name", "department", "salary", "hire_date"])

In [0]:
employee_df.show()

In [0]:
# Fill missing salary with the average salary per department.
# Convert hire_date to date type.
# Add a column years_with_company based on today's date.
# Filter only employees who have been with the company for 3+ years.
# Show average salary by department.

employee_df = (
    employee_df
    .withColumn("salary",
                 when(col("salary").isNull(),
                 avg("salary")
                 .over(Window.partitionBy("department")))
                 .otherwise(col("salary"))
            )
    .withColumn("hire_date", to_date(col("hire_date"), "yyyy-MM-dd"))
    .withColumn("years_with_company", 
                round(months_between(current_date(), col("hire_date")) / lit(12), 0)
                .cast(IntegerType())
            )
    .filter(col("years_with_company") >= 3)
    .groupBy("department")
    .agg(avg("salary").alias("avg_salary"))
    )

In [0]:
employee_df.show()
employee_df.printSchema()

### Challenge 2:

Tasks:

- Clean the price column by removing $ and converting to float.
- Standardize in_stock values to boolean (True/False).
- Fill null prices with the average price.
- Add a new column category based on product name:
- "Laptop", "Tablet", "Phone" → "Mobile Devices"
- "Monitor", "Keyboard" → "Accessories"
- Show sorted products by price descending.



In [0]:
product_df = spark.createDataFrame([
    ("A001", "Laptop", "$999.99", "Yes"),
    ("A002", "Phone", "499.5", "No"),
    ("A003", "Tablet", "$299", "yes"),
    ("A004", "Monitor", "199.99", "NO"),
    ("A005", "Keyboard", None, "Yes")
], ["product_id", "name", "price", "in_stock"])

In [0]:
product_df.show()

In [0]:
# Clean the price column by removing $ and converting to float.
# Standardize in_stock values to boolean (True/False).
# Fill null prices with the average price.
# Add a new column category based on product name:
# "Laptop", "Tablet", "Phone" → "Mobile Devices"
# "Monitor", "Keyboard" → "Accessories"
# Show sorted products by price descending.

window_spec = Window.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

product_df = (
    product_df
    .withColumn("price", regexp_replace(col("price"), "[$]", "").cast("float"))
    .withColumn("in_stock", when(upper(col("in_stock")) == "YES", True).otherwise(False))
    .withColumn("price", when(
        col("price").isNull(),
        avg("price").over(window_spec)
        ).otherwise(col("price"))
    )
    .withColumn("price", round(col("price"), 2))
    .withColumn("category", when(col("name").isin("Laptop", "Tablet", "Phone"), "Mobile Devices")
                .when(col("name").isin("Monitor", "Keyboard"), "Accessories")
            )
    .sort(col("price").desc())
)

In [0]:
product_df.show()
product_df.printSchema()